# Multilingual Embedding Model Evaluation Accelerator
### Databricks-managed Qwen3 versus self-hosted LaBSE on the same synthetic claims

**Decision this notebook supports.** A team with a self-hosted multilingual embedding model needs evidence
that a Databricks-managed model can meet its accuracy requirements before changing architecture. This
notebook runs both candidates on the same labeled multilingual claims and produces a queryable scorecard.
It is an evaluation harness, not a reference architecture that runs both models in production.

**How.** Both candidates receive identical inputs and labels. The evaluation compares:

1. Cross-lingual semantic separation: the same defect in different languages should be closer than different defects.
2. Clustering quality: purity and Adjusted Rand Index against known defect labels.
3. Cross-lingual retrieval: the nearest claim in another language should describe the same defect.
4. Operational fit: Databricks-managed inference through `ai_query` versus a self-hosted open-source runtime.

**Decision rule.** First confirm that each candidate meets the required quality bar. If both do, choose based
on operational requirements such as infrastructure ownership, endpoint availability, and model constraints.
Use only the selected model in production. Never combine the two vector spaces in one index.

> **Preview and GA status.** The core `ai_query` function is GA. The
> `databricks-qwen3-embedding-0-6b` endpoint is currently Public Preview. LaBSE is an Apache-2.0 open model
> loaded through Sentence-Transformers and requires a runtime that the adopter operates.

> **Public evidence boundary.** This repository uses seeded synthetic claims. It proves the evaluation method
> and generates reproducible technical evidence. It does not contain customer data or reproduce a live
> customer system. Optional Day-2 monitoring is a separate job and operational extension.


## 1 · Setup and configuration

The catalog and schema are governed by Unity Catalog. Set the `catalog` and `schema` widgets before running. The catalog defaults to the current catalog when left blank, and all identifiers are safely quoted so workspace-specific names work without editing notebook code.

In [ ]:
# Portable configuration: bundle task parameters and interactive widgets use the same names.
dbutils.widgets.text("catalog", "", "Unity Catalog catalog (blank = current catalog)")
dbutils.widgets.text("schema", "multilingual_warranty_demo", "Target schema")

CATALOG = dbutils.widgets.get("catalog").strip() or spark.sql("SELECT current_catalog()").first()[0]
SCHEMA = dbutils.widgets.get("schema").strip()
if not CATALOG or not SCHEMA:
    raise ValueError("Both catalog and schema must resolve to non-empty names.")

def quote_identifier(value):
    """Quote one Unity Catalog identifier without treating dots as separators."""
    if "\x00" in value:
        raise ValueError("Catalog and schema names cannot contain null bytes.")
    return f"`{value.replace('`', '``')}`"

NAMESPACE = f"{quote_identifier(CATALOG)}.{quote_identifier(SCHEMA)}"
RAW_TBL = f"{NAMESPACE}.`warranty_claims`"
QWEN_TBL = f"{NAMESPACE}.`warranty_claims_qwen3`"
METRICS_TBL = f"{NAMESPACE}.`comparison_metrics`"

QWEN_ENDPOINT = "databricks-qwen3-embedding-0-6b"  # Multilingual, 1024-dim (Public Preview)
LABSE_MODEL = "sentence-transformers/LaBSE"         # Apache-2.0, 768-dim
LABSE_REVISION = "836121a0533e5664b21c7aacc5d22951f2b8b25b"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {NAMESPACE}")
spark.sql(f"USE CATALOG {quote_identifier(CATALOG)}")
spark.sql(f"USE SCHEMA {quote_identifier(SCHEMA)}")
print(f"Ready. Writing to {CATALOG}.{SCHEMA}")

## 2 · Synthetic multilingual warranty corpus
We define 12 physical defect types, each written naturally in English, Korean, Spanish, and German. Every
row carries its true `defect_id` (the ground-truth label) and `language`, so we can measure whether the
embeddings recover the defect grouping across languages. The generator is seeded for reproducibility.

In [ ]:
import random
random.seed(42)

# defect_id -> {language -> [phrasings]}.  Same defect, written naturally in each language.
DEFECTS = {
  "engine_overheat": {
    "en": ["Engine overheating, temperature warning light came on during highway driving.",
           "Coolant temperature spiked and the overheat lamp illuminated.",
           "Motor ran hot and the dashboard temp warning turned red."],
    "ko": ["고속 주행 중 엔진 과열, 온도 경고등이 켜졌습니다.",
           "냉각수 온도가 급상승하고 과열 경고등이 점등되었습니다.",
           "엔진이 과열되어 계기판 온도 경고가 빨간색으로 표시되었습니다."],
    "es": ["Sobrecalentamiento del motor, se encendió la luz de advertencia de temperatura en carretera.",
           "La temperatura del refrigerante subió y se iluminó la luz de sobrecalentamiento.",
           "El motor se calentó y la advertencia de temperatura del tablero se puso roja."],
    "de": ["Motor überhitzt, Temperaturwarnleuchte ging während der Autobahnfahrt an.",
           "Kühlmitteltemperatur stieg stark an und die Überhitzungslampe leuchtete.",
           "Der Motor lief heiß und die Temperaturwarnung im Armaturenbrett wurde rot."],
  },
  "brake_noise": {
    "en": ["Loud grinding noise from front brakes when stopping.",
           "Squealing brakes and vibration through the pedal.",
           "Metallic scraping sound every time I brake."],
    "ko": ["정지할 때 앞 브레이크에서 큰 갈리는 소음이 발생합니다.",
           "브레이크에서 끼익 소리가 나고 페달에 진동이 느껴집니다.",
           "브레이크를 밟을 때마다 금속이 긁히는 소리가 납니다."],
    "es": ["Ruido fuerte de rechinamiento en los frenos delanteros al frenar.",
           "Frenos chirriantes y vibración en el pedal.",
           "Sonido metálico de raspado cada vez que freno."],
    "de": ["Lautes Schleifgeräusch der Vorderbremsen beim Anhalten.",
           "Quietschende Bremsen und Vibration im Pedal.",
           "Metallisches Kratzgeräusch bei jedem Bremsen."],
  },
  "battery_drain": {
    "en": ["Battery drains overnight, car won't start in the morning.",
           "12V battery keeps dying even after a full charge.",
           "Dead battery repeatedly, parasitic drain suspected."],
    "ko": ["배터리가 밤새 방전되어 아침에 시동이 걸리지 않습니다.",
           "완전 충전 후에도 12V 배터리가 계속 방전됩니다.",
           "배터리가 반복적으로 방전되며 암전류가 의심됩니다."],
    "es": ["La batería se descarga durante la noche, el auto no arranca por la mañana.",
           "La batería de 12V se sigue agotando incluso después de una carga completa.",
           "Batería descargada repetidamente, se sospecha de fuga parásita."],
    "de": ["Batterie entlädt sich über Nacht, Auto springt morgens nicht an.",
           "12V-Batterie entlädt sich immer wieder trotz voller Ladung.",
           "Wiederholt leere Batterie, Kriechstrom vermutet."],
  },
  "transmission_slip": {
    "en": ["Transmission slips when shifting from second to third gear.",
           "Gearbox hesitates and RPM flares during acceleration.",
           "Automatic transmission jerks and slips under load."],
    "ko": ["2단에서 3단으로 변속할 때 변속기가 미끄러집니다.",
           "가속 중 기어박스가 머뭇거리고 RPM이 치솟습니다.",
           "자동 변속기가 부하 시 덜컥거리고 미끄러집니다."],
    "es": ["La transmisión patina al cambiar de segunda a tercera marcha.",
           "La caja de cambios duda y las RPM se disparan durante la aceleración.",
           "La transmisión automática da tirones y patina bajo carga."],
    "de": ["Getriebe rutscht beim Schalten vom zweiten in den dritten Gang.",
           "Getriebe zögert und die Drehzahl steigt beim Beschleunigen.",
           "Automatikgetriebe ruckelt und rutscht unter Last."],
  },
  "ac_not_cooling": {
    "en": ["Air conditioning blows warm air, no cold output.",
           "AC stopped cooling, only warm air from vents.",
           "Climate control not cooling the cabin at all."],
    "ko": ["에어컨에서 따뜻한 바람이 나오고 차가운 바람이 안 나옵니다.",
           "에어컨이 냉방이 안 되고 송풍구에서 더운 바람만 나옵니다.",
           "공조 장치가 실내를 전혀 냉방하지 못합니다."],
    "es": ["El aire acondicionado echa aire caliente, no sale frío.",
           "El A/C dejó de enfriar, solo aire caliente por las rejillas.",
           "El climatizador no enfría la cabina en absoluto."],
    "de": ["Klimaanlage bläst warme Luft, keine Kühlung.",
           "Klimaanlage kühlt nicht mehr, nur warme Luft aus den Düsen.",
           "Klimaregelung kühlt den Innenraum überhaupt nicht."],
  },
  "steering_vibration": {
    "en": ["Steering wheel vibrates heavily at highway speeds.",
           "Shaking through the steering wheel above 60 mph.",
           "Front-end wobble and steering vibration when driving fast."],
    "ko": ["고속에서 스티어링 휠이 심하게 진동합니다.",
           "시속 100km 이상에서 핸들이 흔들립니다.",
           "빠르게 주행 시 앞부분 흔들림과 조향 진동이 있습니다."],
    "es": ["El volante vibra fuertemente a velocidades de autopista.",
           "Temblor en el volante por encima de 100 km/h.",
           "Bamboleo del tren delantero y vibración de dirección al conducir rápido."],
    "de": ["Lenkrad vibriert stark bei Autobahngeschwindigkeit.",
           "Zittern im Lenkrad über 100 km/h.",
           "Flattern der Vorderachse und Lenkungsvibration bei schneller Fahrt."],
  },
  "oil_leak": {
    "en": ["Oil leaking under the engine, spots on the garage floor.",
           "Visible engine oil leak near the oil pan gasket.",
           "Losing oil steadily, drips found under the vehicle."],
    "ko": ["엔진 아래에서 오일이 새어 차고 바닥에 자국이 있습니다.",
           "오일 팬 개스킷 근처에서 엔진 오일 누유가 보입니다.",
           "오일이 지속적으로 줄고 차량 아래에서 방울이 발견됩니다."],
    "es": ["Fuga de aceite debajo del motor, manchas en el piso del garaje.",
           "Fuga visible de aceite de motor cerca de la junta del cárter.",
           "Pérdida constante de aceite, se encuentran goteos bajo el vehículo."],
    "de": ["Ölaustritt unter dem Motor, Flecken auf dem Garagenboden.",
           "Sichtbarer Motorölaustritt nahe der Ölwannendichtung.",
           "Stetiger Ölverlust, Tropfen unter dem Fahrzeug gefunden."],
  },
  "infotainment_freeze": {
    "en": ["Infotainment screen freezes and reboots randomly.",
           "Touchscreen becomes unresponsive while driving.",
           "Head unit locks up and restarts on its own."],
    "ko": ["인포테인먼트 화면이 멈추고 무작위로 재부팅됩니다.",
           "주행 중 터치스크린이 반응하지 않습니다.",
           "헤드 유닛이 멈추고 저절로 재시작됩니다."],
    "es": ["La pantalla de infoentretenimiento se congela y se reinicia al azar.",
           "La pantalla táctil deja de responder mientras conduzco.",
           "La unidad central se bloquea y se reinicia sola."],
    "de": ["Infotainment-Bildschirm friert ein und startet zufällig neu.",
           "Touchscreen reagiert während der Fahrt nicht mehr.",
           "Head-Unit hängt sich auf und startet von selbst neu."],
  },
  "door_seal_leak": {
    "en": ["Water leaks into the cabin through the door seal when it rains.",
           "Wet carpet on the passenger side after heavy rain.",
           "Wind noise and water intrusion around the door weatherstrip."],
    "ko": ["비가 올 때 도어 씰을 통해 실내로 물이 샙니다.",
           "폭우 후 조수석 카펫이 젖어 있습니다.",
           "도어 웨더스트립 주변에서 바람 소리와 물 침투가 있습니다."],
    "es": ["Entra agua a la cabina por el sello de la puerta cuando llueve.",
           "Alfombra mojada del lado del pasajero tras lluvia fuerte.",
           "Ruido de viento e infiltración de agua alrededor del burlete de la puerta."],
    "de": ["Wasser dringt bei Regen durch die Türdichtung in den Innenraum.",
           "Nasser Teppich auf der Beifahrerseite nach starkem Regen.",
           "Windgeräusche und Wassereintritt an der Türdichtung."],
  },
  "check_engine_light": {
    "en": ["Check engine light on, code points to a misfire.",
           "MIL illuminated with rough idle and hesitation.",
           "Engine warning light stays on, car runs rough."],
    "ko": ["체크 엔진 경고등이 켜지고 코드가 실화를 나타냅니다.",
           "엔진 경고등이 점등되고 공회전이 거칠며 머뭇거립니다.",
           "엔진 경고등이 계속 켜져 있고 차가 거칠게 작동합니다."],
    "es": ["Luz de check engine encendida, el código indica una falla de encendido.",
           "Testigo MIL encendido con ralentí irregular y titubeo.",
           "La luz de advertencia del motor sigue encendida, el auto funciona mal."],
    "de": ["Motorkontrollleuchte an, Code deutet auf Fehlzündung hin.",
           "MIL leuchtet bei unrundem Leerlauf und Zögern.",
           "Motorwarnleuchte bleibt an, Fahrzeug läuft unrund."],
  },
  "paint_defect": {
    "en": ["Paint peeling and bubbling on the hood after six months.",
           "Clear coat flaking off on the roof panel.",
           "Visible paint chipping and discoloration on the door."],
    "ko": ["6개월 후 후드의 페인트가 벗겨지고 부풀어 오릅니다.",
           "루프 패널의 클리어 코트가 벗겨지고 있습니다.",
           "도어에 페인트 벗겨짐과 변색이 보입니다."],
    "es": ["La pintura se pela y burbujea en el capó después de seis meses.",
           "El barniz se descascara en el panel del techo.",
           "Astillado de pintura y decoloración visible en la puerta."],
    "de": ["Lack blättert ab und wirft Blasen auf der Motorhaube nach sechs Monaten.",
           "Klarlack blättert am Dachblech ab.",
           "Sichtbare Lackabplatzungen und Verfärbung an der Tür."],
  },
  "seatbelt_fault": {
    "en": ["Seatbelt won't retract and stays loose.",
           "Driver seatbelt latch fails to click and hold.",
           "Seatbelt warning chimes even when buckled."],
    "ko": ["안전벨트가 되감기지 않고 느슨한 상태로 유지됩니다.",
           "운전석 안전벨트 버클이 제대로 잠기지 않습니다.",
           "안전벨트를 착용해도 경고음이 울립니다."],
    "es": ["El cinturón de seguridad no se retrae y queda flojo.",
           "El broche del cinturón del conductor no ajusta ni sujeta.",
           "El aviso del cinturón suena aunque esté abrochado."],
    "de": ["Sicherheitsgurt zieht sich nicht ein und bleibt locker.",
           "Gurtschloss des Fahrers rastet nicht ein.",
           "Gurtwarnung ertönt trotz angelegtem Gurt."],
  },
}

LANGS = ["en", "ko", "es", "de"]
rows, cid = [], 1000
for defect_id, by_lang in DEFECTS.items():
    for lang in LANGS:
        for text in by_lang[lang]:
            rows.append((f"WC-{cid}", defect_id, lang, text))
            cid += 1
# light shuffle so tables aren't grouped by defect/language on disk
random.shuffle(rows)
print(f"Generated {len(rows)} synthetic claims across {len(DEFECTS)} defects x {len(LANGS)} languages")

### Persist the reproducible corpus

The generated rows are written to a managed Delta table so both embedding paths and the companion Day-2 notebook read the identical inputs.

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType

schema = StructType([
    StructField("claim_id",  StringType(), False),
    StructField("defect_id", StringType(), False),   # ground-truth label (synthetic only)
    StructField("language",  StringType(), False),
    StructField("claim_text",StringType(), False),
])
df = spark.createDataFrame(rows, schema)
df.write.mode("overwrite").option("overwriteSchema","true").saveAsTable(RAW_TBL)
print(f"Wrote {df.count()} rows to {RAW_TBL}")
display(spark.sql(f"SELECT language, count(*) AS claims FROM {RAW_TBL} GROUP BY language ORDER BY language"))

## 3 · Batch-embed with Qwen3 via `ai_query`
This is the recommended, generally available batch path. `ai_query` calls the managed Foundation Model API
endpoint, auto-parallelizes over the corpus, and returns a 1024-dim vector per row. There is no endpoint to
host, size, or warm up.

In [ ]:
spark.sql(f"""
  CREATE OR REPLACE TABLE {QWEN_TBL} AS
  SELECT claim_id, defect_id, language, claim_text,
         ai_query('{QWEN_ENDPOINT}', claim_text) AS qwen3_embedding
  FROM {RAW_TBL}
""")
n = spark.sql(f"SELECT count(*) c, min(size(qwen3_embedding)) d FROM {QWEN_TBL}").collect()[0]
print(f"Qwen3 embeddings written for {n['c']} rows, dimension = {n['d']}")
display(spark.sql(f"SELECT claim_id, language, defect_id, size(qwen3_embedding) AS dim FROM {QWEN_TBL} LIMIT 5"))

## 4 · Embed the same corpus with LaBSE (Sentence-Transformers)

LaBSE is the open, self-hosted path. We install pinned client dependencies and load a pinned Hugging Face model revision, then embed the identical claim texts. The workspace needs outbound access to `huggingface.co`; see `SETUP.md` for an air-gapped alternative.

In [ ]:
# LaBSE is open and does not require a Hugging Face token.
# Versions are pinned so a clean clone does not silently change its local evaluation stack.
%pip install -q "sentence-transformers==5.1.1" "scikit-learn==1.7.2" "threadpoolctl==3.6.0"
dbutils.library.restartPython()

### Restore configuration after the Python restart

Databricks preserves widget values across `restartPython()`, but Python variables are cleared. This cell safely reconstructs the selected namespace and loads the pinned LaBSE revision.

In [ ]:
# Python restarts after %pip, so restore widget-backed configuration explicitly.
CATALOG = dbutils.widgets.get("catalog").strip() or spark.sql("SELECT current_catalog()").first()[0]
SCHEMA = dbutils.widgets.get("schema").strip()
if not CATALOG or not SCHEMA:
    raise ValueError("Both catalog and schema must resolve to non-empty names.")

def quote_identifier(value):
    if "\x00" in value:
        raise ValueError("Catalog and schema names cannot contain null bytes.")
    return f"`{value.replace('`', '``')}`"

NAMESPACE = f"{quote_identifier(CATALOG)}.{quote_identifier(SCHEMA)}"
RAW_TBL = f"{NAMESPACE}.`warranty_claims`"
QWEN_TBL = f"{NAMESPACE}.`warranty_claims_qwen3`"
METRICS_TBL = f"{NAMESPACE}.`comparison_metrics`"
LABSE_MODEL = "sentence-transformers/LaBSE"
LABSE_REVISION = "836121a0533e5664b21c7aacc5d22951f2b8b25b"

import numpy as np
from sentence_transformers import SentenceTransformer

pdf = spark.table(RAW_TBL).toPandas().sort_values("claim_id").reset_index(drop=True)
labse = SentenceTransformer(LABSE_MODEL, revision=LABSE_REVISION)
labse_emb = labse.encode(pdf["claim_text"].tolist(), normalize_embeddings=True, show_progress_bar=False)
print(f"LaBSE embeddings: {labse_emb.shape[0]} rows, dimension = {labse_emb.shape[1]}")

### Align managed and open-model vectors

The next step joins Qwen3 vectors to the same deterministic claim ordering used for LaBSE and normalizes them before cosine-based evaluation.

In [ ]:
# Pull the Qwen3 vectors back aligned to the same claim order
qpdf = spark.table(QWEN_TBL).toPandas().set_index("claim_id")
qpdf = qpdf.loc[pdf["claim_id"].values]
qwen_emb = np.array(qpdf["qwen3_embedding"].tolist(), dtype=float)
# normalize Qwen3 vectors for cosine geometry
qwen_emb = qwen_emb / np.linalg.norm(qwen_emb, axis=1, keepdims=True)
labels    = pdf["defect_id"].values
languages = pdf["language"].values
print("Aligned. Qwen3:", qwen_emb.shape, "| LaBSE:", labse_emb.shape)

## 5 · Cross-lingual proof
The core test: for a good multilingual embedder, the same defect written in different languages should be
closer than two different defects. Per model, we compute:
- **Cross-lingual same-defect similarity**: average cosine between claims that share a defect but are in *different* languages (should be high).
- **Different-defect similarity**: average cosine between claims of *different* defects (should be lower).
- **Separation margin**: the gap between the two. Bigger is better.

In [ ]:
from itertools import combinations

def cos(a, b): return float(np.dot(a, b))

def separation(emb):
    same_cross, diff = [], []
    n = len(emb)
    for i, j in combinations(range(n), 2):
        s = cos(emb[i], emb[j])
        if labels[i] == labels[j] and languages[i] != languages[j]:
            same_cross.append(s)          # same defect, different language
        elif labels[i] != labels[j]:
            diff.append(s)                # different defect
    sc, df_ = float(np.mean(same_cross)), float(np.mean(diff))
    return sc, df_, sc - df_

separation_results = {}
for model_key, emb in [("qwen3_managed", qwen_emb), ("labse_self_hosted", labse_emb)]:
    same_similarity, different_similarity, margin = separation(emb)
    separation_results[model_key] = {
        "same_defect_similarity": same_similarity,
        "different_defect_similarity": different_similarity,
        "separation_margin": margin,
    }
    print(
        f"{model_key:18s} | cross-lingual same-defect cos = {same_similarity:.3f} | "
        f"different-defect cos = {different_similarity:.3f} | margin = {margin:.3f}"
    )

## 6 · Clustering quality comparison
We cluster each model's vectors into *k = number of defects* groups and measure **purity** (how cleanly each
cluster maps to a single true defect) and **Adjusted Rand Index** against the ground-truth labels. This is
the "does it recover the defect structure across languages" test, scored on identical data for both models.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score
import pandas as pd

k = len(set(labels))
uniq = sorted(set(labels)); lab_idx = np.array([uniq.index(l) for l in labels])

def purity(true_idx, pred):
    total = 0
    for c in set(pred):
        members = true_idx[pred == c]
        total += np.bincount(members).max()
    return total / len(true_idx)

results = []
for model_key, display_name, hosting_model, emb in [
    ("qwen3_managed", "Qwen3", "databricks_managed", qwen_emb),
    ("labse_self_hosted", "LaBSE", "self_hosted", labse_emb),
]:
    km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(emb)
    results.append({
        "model_key": model_key,
        "model": display_name,
        "hosting_model": hosting_model,
        "dimension": int(emb.shape[1]),
        "cluster_purity": float(purity(lab_idx, km.labels_)),
        "adjusted_rand_index": float(adjusted_rand_score(lab_idx, km.labels_)),
    })
comp = pd.DataFrame(results)
display(spark.createDataFrame(comp))

## 7 · Cross-lingual retrieval and decision scorecard
For every claim, is its nearest neighbor *in a different language* the same defect? This directly measures
whether reports of the same problem stay connected across regions. The notebook combines this result with
semantic separation and clustering metrics, then persists one queryable `comparison_metrics` scorecard in
Unity Catalog. That table is the durable evidence used to compare candidates and make the model decision.

In [ ]:
def crosslingual_retrieval_acc(emb):
    sim = emb @ emb.T
    np.fill_diagonal(sim, -1.0)
    correct = 0
    for i in range(len(emb)):
        # Restrict candidates to a different language.
        candidates = np.where(languages != languages[i])[0]
        nearest = candidates[np.argmax(sim[i, candidates])]
        if labels[nearest] == labels[i]:
            correct += 1
    return correct / len(emb)

retrieval_results = {
    "qwen3_managed": crosslingual_retrieval_acc(qwen_emb),
    "labse_self_hosted": crosslingual_retrieval_acc(labse_emb),
}

scorecard_rows = []
for result in results:
    model_key = result["model_key"]
    scorecard_rows.append({
        **result,
        **separation_results[model_key],
        "crosslingual_retrieval_accuracy": float(retrieval_results[model_key]),
        "corpus_rows": int(len(labels)),
        "defect_count": int(len(set(labels))),
        "language_count": int(len(set(languages))),
    })

from pyspark.sql import functions as F

scorecard = spark.createDataFrame(pd.DataFrame(scorecard_rows)).withColumn(
    "evaluated_at", F.current_timestamp()
)
scorecard.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(METRICS_TBL)
print(f"Evaluation scorecard written to {METRICS_TBL}")
display(scorecard.orderBy("model_key"))

## 8 · Interpret the selected model's clusters
The scorecard supports the model choice. This final evaluation view shows how the managed candidate's
clusters can be interpreted as claim volume by dominant defect and language coverage. It is a diagnostic
view of the synthetic evaluation corpus, not a production alert or a claim about the customer's runtime.

In [ ]:
km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(qwen_emb)
insight = pd.DataFrame({"cluster": km.labels_, "defect_id": labels, "language": languages})
# Name each cluster by its dominant true defect, then show cross-language volume.
cluster_name = insight.groupby("cluster")["defect_id"].agg(lambda s: s.value_counts().index[0])
insight["dominant_defect"] = insight["cluster"].map(cluster_name)
summary = (insight.groupby("dominant_defect")
           .agg(claims=("defect_id", "size"), languages=("language", "nunique"))
           .reset_index().sort_values("claims", ascending=False))
print("Evaluation view: claim volume per managed-model cluster and language coverage")
display(spark.createDataFrame(summary))

## 9 · Decision summary

- Both **Qwen3-Embedding** (Databricks-managed, 1024-dim) and **LaBSE** (self-hosted OSS, 768-dim) are tested
  on identical labeled claims.
- The `comparison_metrics` table makes semantic separation, clustering quality, and cross-lingual retrieval
  queryable as one decision scorecard.
- If both candidates meet the adopter's accuracy bar, the remaining decision is operational: use Qwen3 to
  avoid hosting model infrastructure, or use LaBSE when an open model under adopter control is required.
- Production uses one selected model and one vector space. It does not run this head-to-head evaluation as
  the normal serving path.

**Optional operational extension.** The separate monitoring job demonstrates how a team can score later
batches against prior history and enforce quality thresholds after selecting a model. Its synthetic results
are guidance, not evidence of a customer production implementation.

*All repository results use synthetic data and are reproducible from the seeded corpus.*
